# Fundamentos de PySpark para Data Engineering

Este notebook explica conceptos básicos de PySpark usando un caso simple de datos meteorológicos tipo Open-Meteo.

## Objetivos

Al finalizar, debes poder explicar y practicar:

1. Cómo leer archivos CSV, JSON y Parquet.
2. Qué son transformaciones **narrow** y **wide**.
3. Cómo crear columnas con `select`, `withColumn` y `withColumns`.
4. Diferencias de performance al crear muchas columnas.
5. Cómo usar `groupBy`.
6. Cómo hacer joins: `inner`, `left` y `left_anti`.
7. Cómo crear una tabla Delta.
8. Diferencias entre SCD Type 0, SCD Type 1 y SCD Type 2.


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

username = spark.sql("SELECT current_user()").collect()[0][0]
CATALOG = 'taller'
SCHEMA = 'openmeteo'
VOLUMEN = f'/Volumes/{CATALOG}/{SCHEMA}/data'


In [0]:
spark.sql(f'CREATE CATALOG IF NOT EXISTS {CATALOG}')
spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SCHEMA}')
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.data COMMENT 'Volumen para guardar archivos del taller';")

# Escogiendo la Herramienta adecuada

## Pandas vs PySpark

| Criterio | Pandas | PySpark |
|---|---|---|
| Volumen de datos | Hasta unos GB (memoria local) | Desde MB hasta TB/masivo (distribuido) |
| Facilidad de uso | Muy fácil, sintaxis simple | Similar a Pandas, pero requiere cluster |
| Performance | Rápido en datos pequeños | Escala horizontalmente, mejor en datos grandes |
| Integración con Spark | No | Sí, nativo |
| Casos ideales | Análisis exploratorio, prototipos, notebooks | ETL, procesamiento masivo, pipelines productivos |
| Recursos necesarios | Laptop/servidor local | Cluster Spark (Databricks, EMR, etc.) |


%md 
## Ventaja de PySpark: Lazy Evaluation vs Pandas

Una de las ventajas importantes de PySpark frente a Pandas es el uso de **lazy evaluation**.

En Pandas, cuando se lee un archivo o se ejecuta una transformación, normalmente la operación se ejecuta de inmediato y los datos se cargan en la memoria de una sola máquina. Esto funciona muy bien con datasets pequeños o medianos, pero puede convertirse en un problema cuando el volumen de datos crece.

En PySpark, muchas operaciones no se ejecutan inmediatamente. Spark primero construye un **plan lógico** con las transformaciones que se quieren aplicar. La ejecución real ocurre solamente cuando se llama una **acción**, como `show()`, `count()`, `write()`, `collect()` o `display()`.

### Comparación rápida

| Aspecto | Pandas | PySpark |
|---|---|---|
| Ejecución | Inmediata | Lazy evaluation |
| Memoria | Carga datos en memoria local | Procesa datos de forma distribuida |
| Optimización | Limitada al proceso local | Spark optimiza el plan antes de ejecutar |
| Escalabilidad | Depende de una sola máquina | Puede usar múltiples nodos |
| Mejor para | Análisis pequeño o prototipos | Pipelines productivos y grandes volúmenes |

In [0]:
import os
import gc
import psutil

def show_driver_memory(label: str = ""):
    process = psutil.Process(os.getpid())
    memory_mb = process.memory_info().rss / 1024 / 1024

    print(f"{label} | Driver memory used: {memory_mb:,.2f} MB")

## 1. Creando datos de ejemplo



## 2. Leer archivos con PySpark

En PySpark normalmente lees archivos usando:

```python
spark.read.format(...).option(...).load(path)
```

Formatos comunes:

- CSV: útil para intercambio, pero no conserva tipos de datos de forma nativa.
- JSON: flexible, común para APIs, pero puede ser costoso para grandes volúmenes.
- Parquet: columnar, comprimido y eficiente para analítica.
- Delta: Parquet + transaction log + capacidades ACID.

### Buenas prácticas al leer archivos

| Práctica | Por qué importa |
|---|---|
| Definir schema explícito | Evita inferencia lenta y cambios inesperados de tipos |
| Separar raw/bronze/silver/gold | Facilita trazabilidad y calidad |
| Evitar leer columnas innecesarias | Permite projection pruning |
| Filtrar temprano | Reduce volumen procesado |

## 3. Transformaciones narrow vs wide

En Spark, una transformación puede ser **narrow** o **wide**.

### Narrow transformation

Una transformación es narrow cuando cada partición de salida depende de una sola partición de entrada.

Ejemplos:

- `select`
- `filter`
- `withColumn`
- `drop`
- `cast`

Normalmente no requiere `shuffle`.

### Wide transformation

Una transformación es wide cuando Spark necesita mover datos entre particiones. Eso se llama **shuffle**.

Ejemplos:

- `groupBy`
- `join` cuando las llaves no están co-particionadas
- `distinct`
- `orderBy`
- `repartition`

El shuffle suele ser una de las operaciones más costosas en Spark porque implica red, disco, serialización y redistribución de datos.

## 4. Crear columnas: `select`, `withColumn` y `withColumns`

Existen varias formas de crear o transformar columnas.

### `select`

Es ideal cuando quieres elegir columnas y crear varias expresiones en un solo paso.

### `withColumn`

Es cómodo para agregar o reemplazar una columna individual.

### `withColumns`

Permite crear varias columnas a la vez usando un diccionario. Es útil para evitar muchos `withColumn` encadenados.

> Nota de performance: Spark suele optimizar varios `Project` con Catalyst, pero muchos `withColumn` en loops grandes pueden aumentar el tiempo de planificación y crear planes lógicos más complejos. Para muchas columnas, suele ser mejor `select` o `withColumns`.

### Comparación práctica

| Método | Mejor uso | Comentario de performance |
|---|---|---|
| `select` | Seleccionar y crear varias columnas en un solo paso | Muy claro y eficiente para proyecciones grandes |
| `withColumn` | Agregar o reemplazar una columna puntual | Bueno para pocos cambios; evitar loops muy largos |
| `withColumns` | Agregar muchas columnas a la vez | Reduce encadenamiento y mejora legibilidad |

Regla práctica: para una o dos columnas, `withColumn` está bien. Para muchas columnas, prefiere `select` o `withColumns`.

## 5. `groupBy`

`groupBy` se usa para agregaciones. Es una transformación wide porque normalmente requiere shuffle.

Ejemplos comunes:

- promedio de temperatura por ubicación
- total de lluvia por provincia
- cantidad de eventos por día

## 6. Joins: `inner`, `left` y `left_anti`

Usaremos `weather_df` como tabla de hechos y `location_df` como tabla de referencia.

### `inner join`

Devuelve solo registros con match en ambos lados.

### `left join`

Devuelve todos los registros de la izquierda y agrega columnas de la derecha cuando hay match.

### `left_anti join`

Devuelve registros de la izquierda que **no tienen match** en la derecha. Es muy útil para validaciones de calidad.

### Buenas prácticas para joins

| Situación | Recomendación |
|---|---|
| Tabla pequeña de referencia | Usar broadcast join si aplica |
| Duplicados en la llave de join | Deduplicar antes del join para evitar explosión de filas |
| Validación de integridad | Usar `left_anti` para detectar registros sin referencia |
| Joins grandes | Revisar particionamiento, skew y estadísticas |


## 8. SCD Type 0 vs SCD Type 1 vs SCD Type 2

SCD significa **Slowly Changing Dimension**. Se usa para manejar cambios en dimensiones o catálogos.

| Tipo | Qué hace | Cuándo usarlo |
|---|---|---|
| SCD Type 0 | No actualiza atributos existentes | Datos que no deberían cambiar, como fecha de nacimiento o código original |
| SCD Type 1 | Sobrescribe el valor anterior | Catálogos donde solo importa el estado actual |
| SCD Type 2 | Guarda historial de versiones | Cuando necesitas saber cómo era el dato en el pasado |

In [0]:
initial_locations = (
    spark.createDataFrame([
        ("PTY", "Panama City", "Panama", "Panama", 8.9824, -79.5199, "2026-06-01 00:00:00"),
        ("DAV", "David", "Panama", "Chiriqui", 8.4273, -82.4308, "2026-06-01 00:00:00"),
    ], ["location_id", "location_name", "country", "province", "latitude", "longitude", "updated_at"])
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

incoming_locations = (
    spark.createDataFrame([
        ("PTY", "Ciudad de Panama", "Panama", "Panama", 8.9824, -79.5199, "2026-06-10 00:00:00"), # cambio de nombre
        ("DAV", "David", "Panama", "Chiriqui", 8.4273, -82.4308, "2026-06-10 00:00:00"),          # sin cambio
        ("BOC", "Bocas del Toro", "Panama", "Bocas del Toro", 9.3403, -82.2420, "2026-06-10 00:00:00"), # nuevo
    ], ["location_id", "location_name", "country", "province", "latitude", "longitude", "updated_at"])
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)


### 8.1 SCD Type 0: no actualizar atributos existentes

En SCD0, si llega una versión nueva de una llave ya existente, se ignora. Solo se insertan llaves nuevas.

In [0]:
scd0_path = f"{VOLUMEN}/delta/scd0"
try:
    dbutils.fs.rm(scd0_path, True)
except Exception:
    pass

initial_locations.write.format("delta").mode("overwrite").save(scd0_path)
spark.sql(f"select * FROM delta.`{scd0_path}`").display()


In [0]:
existing_scd0_df = spark.read.format("delta").load(scd0_path).select("location_id")
new_only_df = incoming_locations.join(existing_scd0_df, on="location_id", how="left_anti")

new_only_df.write.format("delta").mode("append").save(scd0_path)

print("Resultado SCD0: PTY no cambia, BOC se inserta")
spark.sql(f"select * FROM delta.`{scd0_path}`").display()

### 8.2 SCD Type 1: sobrescribir atributos

En SCD1, si una llave ya existe, se actualizan sus atributos. No conserva historial.

In [0]:
scd1_path = f"{VOLUMEN}/delta/scd1_location"
try:
    dbutils.fs.rm(scd1_path, True)
except Exception:
    pass

initial_locations.write.format("delta").mode("overwrite").save(scd1_path)
spark.read.format("delta").load(scd1_path).orderBy("location_id").display()

In [0]:

(
    DeltaTable.forPath(spark, scd1_path)
    .alias("target")
    .merge(
        incoming_locations.alias("source"),
        "target.location_id = source.location_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Resultado SCD1: PTY se actualiza, BOC se inserta")
spark.read.format("delta").load(scd1_path).orderBy("location_id").display()

### 8.3 SCD Type 2: guardar historial

En SCD2 se manejan versiones usando columnas como:

- `start_date`
- `end_date`
- `is_current`
- `hash_diff`

Cuando cambia un atributo, se cierra la versión anterior y se inserta una nueva versión vigente.

In [0]:
def add_scd2_columns(df):
    business_cols = ["location_name", "country", "province", "latitude", "longitude"]
    return (
        df
        .withColumn(
            "hash_diff",
            F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in business_cols]), 256)
        )
        .withColumn("start_date", F.col("updated_at"))
        .withColumn("end_date", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
    )

scd2_path = f"{VOLUMEN}/delta/scd2_location"
scd2_table = "scd2_location"
try:
    dbutils.fs.rm(scd2_path, True)
except Exception:
    pass


initial_scd2_df = add_scd2_columns(initial_locations)
initial_scd2_df.write.format("delta").mode("overwrite").save(scd2_path)
spark.read.format('delta').load(scd2_path).display()

In [0]:


incoming_scd2_df = add_scd2_columns(incoming_locations)
current_scd2_df = spark.read.format("delta").load(scd2_path).where("is_current = true")

# Detecta registros nuevos o cambiados.
changes_df = (
    incoming_scd2_df.alias("source")
    .join(current_scd2_df.alias("target"), on="location_id", how="left")
    .where("target.location_id IS NULL OR source.hash_diff <> target.hash_diff")
    .select("source.*")
)



In [0]:
incoming_scd2_df.display()

In [0]:
changes_df.display()

In [0]:
# 1. Cerrar versiones actuales que cambiaron.
DeltaTable.forPath(spark, scd2_path).alias("target").merge(
    changes_df.alias("source"),
    "target.location_id = source.location_id AND target.is_current = true AND target.hash_diff <> source.hash_diff"
).whenMatchedUpdate(set={
    "end_date": "source.start_date",
    "is_current": "false"
}).execute()

# 2. Insertar las nuevas versiones y los registros nuevos.
changes_df.write.format("delta").mode("append").save(scd2_path)

print("Resultado SCD2: PTY conserva versión anterior y nueva; BOC se inserta")
display(
    spark.read.format("delta").load(scd2_path)
    .orderBy("location_id", "start_date")
)

In [0]:
spark.read.format("delta").load(scd2_path).withColumn('file_name',F.col('_metadata.file_path'))\
.select('file_name').distinct().display()

## Resumen del notebook

| Concepto | Idea clave |
|---|---|
| Leer archivos | Usar `spark.read`; preferir schema explícito y formatos columnares |
| Narrow transformation | No requiere mover datos entre particiones |
| Wide transformation | Requiere shuffle; suele ser más costosa |
| `select` | Mejor para proyecciones y muchas columnas nuevas |
| `withColumn` | Cómodo para una columna puntual |
| `withColumns` | Mejor para agregar varias columnas juntas |
| `groupBy` | Agregación con shuffle |
| `join` | Combina datasets; revisar duplicados, skew y broadcast |
| `left_anti` | Excelente para validaciones de calidad |
| Tabla Delta | Tabla ACID sobre Parquet con transaction log |
| SCD0 | No cambia atributos existentes |
| SCD1 | Sobrescribe valores |
| SCD2 | Conserva historial de cambios |